# 05 — Export, Validate, and Optionally Publish Dataset

**Purpose:** materialize the local Hugging Face package under `NCM Research/huggingface/`, validate table contents/checksums, and optionally upload it to the Hub.

Local population happens automatically. Remote upload defaults to **off** and requires an explicit license-review acknowledgement. The public package excludes raw XSum source text and reference summaries, but generated outputs can still have upstream terms; verify those terms before publication.

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib
import json

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
PROJECT_PARENT = Path("/content/drive/MyDrive/NCM Research")
PROJECT_ROOT = PROJECT_PARENT / "github"
HF_ROOT = PROJECT_PARENT / "huggingface"
CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"

# --------------------------------------------------
# 2. Ensure Drive is mounted
# --------------------------------------------------
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# --------------------------------------------------
# 3. Validate project structure
# --------------------------------------------------
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Repository not found: {PROJECT_ROOT}")

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"Missing pyproject.toml in {PROJECT_ROOT}"
    )

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"Experiment config not found: {CONFIG_PATH}"
    )

HF_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 4. Register the existing local package
# --------------------------------------------------
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--no-build-isolation",
        "-e", str(PROJECT_ROOT),
    ],
    check=True,
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
importlib.invalidate_caches()

# --------------------------------------------------
# 5. Export and validate
# --------------------------------------------------
from ncm_factuality.config import load_config, resolve_paths
from ncm_factuality.dataset_export import (
    build_huggingface_dataset,
    validate_huggingface_package,
)

cfg = load_config(CONFIG_PATH)
paths = resolve_paths(PROJECT_ROOT, cfg)
run_dir = Path(paths["run_dir"])

print(f"Project root: {PROJECT_ROOT}")
print(f"Run directory: {run_dir}")
print(f"Dataset package destination: {HF_ROOT}")
print("Building local dataset package...")

manifest = build_huggingface_dataset(
    PROJECT_PARENT,
    run_dir,
    cfg,
)

print("Validating local dataset package...")
validation = validate_huggingface_package(HF_ROOT)

result = {
    "local_export": manifest,
    "validation": {
        "valid": validation.get("valid", False),
        "row_tables": list(manifest.get("tables", {}).keys()),
    },
}

print(json.dumps(result, indent=2, default=str))

if not validation.get("valid", False):
    raise RuntimeError(
        "Hugging Face dataset package validation failed. "
        "Review the validation output before attempting publication."
    )

print("\nLocal dataset export and validation completed successfully.")

## Optional remote upload — leave disabled until you are ready

Before making anything public, review `huggingface/README.md`, `huggingface/LICENSE`, `docs/LICENSING_AND_DATA_GOVERNANCE.md`, the official NCM terms, the XSum data terms, and whether generated text can be redistributed under your intended use. Never upload the `private/` directory or paste an access token into a notebook cell.

When publication has been cleared, authenticate interactively with `hf auth login` or a Colab Secret, set the target repo ID below, and explicitly change both flags to `True`. The publisher validates the manifest and checksums first.

In [ ]:
# Deliberate opt-in only.
UPLOAD_TO_HF = False
LICENSE_REVIEW_CONFIRMED = False
HF_DATASET_REPO_ID = ""  # e.g. your-account/ncm-hierarchy-factuality-eval

PROJECT_PARENT = Path("/content/drive/MyDrive/NCM Research")
HF_ROOT = PROJECT_PARENT / "huggingface"

if UPLOAD_TO_HF:
    if not LICENSE_REVIEW_CONFIRMED:
        raise PermissionError(
            "Confirm the licensing review only after reviewing "
            "all third-party code, model, and dataset terms."
        )

    if not HF_DATASET_REPO_ID or "/" not in HF_DATASET_REPO_ID:
        raise ValueError(
            "Set HF_DATASET_REPO_ID to your-account/repository-name."
        )

    from ncm_factuality.huggingface_publish import push_to_hub

    print(
        push_to_hub(
            HF_ROOT,
            HF_DATASET_REPO_ID,
            confirm_license_review=True,
            create_repo=False,
        )
    )
else:
    print(
        f"Local package is at {HF_ROOT}. "
        "Remote upload was intentionally not performed."
    )